# 🗄️ Amazon India — SQL Database Integration (SQLite)

| Step | Task |
|------|------|
| 1 | Setup & Imports |
| 2 | Connect to SQLite Database |
| 3 | Load Master Cleaned CSV |
| 4 | Build 4 Normalised DataFrames |
| 5 | Create Schema & Load Tables |
| 6 | Create Indexes for Performance |
| 7 | Validation & Row Count Checks |
| 8 | Analytics Queries (Dashboard KPIs) |
| 9 | Connection Test for Streamlit |

## ① Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import time
from sqlalchemy import create_engine, text

# ── Path config — update these to match your folder layout ───────────────────
MASTER_CSV   = '../data/cleaned/amazon_india_master_cleaned.csv'
PRODUCTS_CSV = '../data/amazon_india_products_catalog.csv'
DB_PATH      = '../data/amazon_india.db'    # SQLite DB output

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

print('✅ Libraries loaded.')
print(f'   Master CSV : {MASTER_CSV}')
print(f'   DB output  : {DB_PATH}')

✅ Libraries loaded.
   Master CSV : ../data/cleaned/amazon_india_master_cleaned.csv
   DB output  : ../data/amazon_india.db


## ② Connect to SQLite Database

> No server needed — SQLite creates the `.db` file automatically

In [2]:
# ── Create SQLAlchemy engine ──────────────────────────────────────────────────
engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)

# ── Enable WAL mode + foreign keys for better performance ─────────────────────
with engine.connect() as conn:
    conn.execute(text('PRAGMA journal_mode=WAL'))
    conn.execute(text('PRAGMA foreign_keys=ON'))
    conn.execute(text('PRAGMA cache_size=-64000'))   # 64MB cache
    conn.execute(text('PRAGMA synchronous=NORMAL'))
    version = conn.execute(text('SELECT sqlite_version()')).fetchone()[0]

print('✅ SQLite connected!')
print(f'   Version  : {version}')
print(f'   DB Path  : {os.path.abspath(DB_PATH)}')
print(f'   Mode     : WAL (Write-Ahead Logging) enabled')

✅ SQLite connected!
   Version  : 3.45.1
   DB Path  : /home/ganesaperumal/Documents/DS/amazon_india_analytics/data/amazon_india.db
   Mode     : WAL (Write-Ahead Logging) enabled


## ③ Load Master Cleaned CSV

In [3]:
# ── Load master cleaned data ──────────────────────────────────────────────────
print('Loading master cleaned CSV...')
t0 = time.time()
master_df = pd.read_csv(MASTER_CSV, low_memory=False)
master_df['order_date'] = pd.to_datetime(master_df['order_date'], errors='coerce')
print(f'✅ Master loaded in {time.time()-t0:.1f}s')
print(f'   Shape   : {master_df.shape[0]:,} rows × {master_df.shape[1]} columns')
print(f'   Memory  : {master_df.memory_usage(deep=True).sum()/1e6:.1f} MB')
print(f'   Years   : {master_df["order_date"].dt.year.min()} → {master_df["order_date"].dt.year.max()}')
print()

# ── Load products catalog ─────────────────────────────────────────────────────
print('Loading products catalog...')
products_df = pd.read_csv(PRODUCTS_CSV, low_memory=False)
print(f'✅ Products loaded : {products_df.shape[0]:,} rows × {products_df.shape[1]} columns')
print()
print('Columns in master_df:')
for i, col in enumerate(master_df.columns.tolist(), 1):
    print(f'  {i:2}. {col}')

Loading master cleaned CSV...
✅ Master loaded in 8.6s
   Shape   : 1,122,000 rows × 34 columns
   Memory  : 1186.6 MB
   Years   : 2015 → 2025

Loading products catalog...
✅ Products loaded : 2,004 rows × 11 columns

Columns in master_df:
   1. transaction_id
   2. order_date
   3. customer_id
   4. product_id
   5. product_name
   6. category
   7. subcategory
   8. brand
   9. original_price_inr
  10. discount_percent
  11. discounted_price_inr
  12. quantity
  13. subtotal_inr
  14. delivery_charges
  15. final_amount_inr
  16. customer_city
  17. customer_state
  18. customer_tier
  19. customer_spending_tier
  20. customer_age_group
  21. payment_method
  22. delivery_days
  23. delivery_type
  24. is_prime_member
  25. is_festival_sale
  26. festival_name
  27. customer_rating
  28. return_status
  29. order_month
  30. order_year
  31. order_quarter
  32. product_weight_kg
  33. is_prime_eligible
  34. product_rating


## ④ Build 4 Normalised DataFrames

```
┌─────────────────┐     ┌──────────────────┐
│   customers     │     │    products      │
│  customer_id PK │     │  product_id   PK │
│  city/state     │     │  name/category   │
│  tier/age       │     │  brand/rating    │
│  is_prime       │     │  is_prime_elig   │
└────────┬────────┘     └────────┬─────────┘
         └──────────┬────────────┘
                    │
          ┌─────────▼──────────┐     ┌──────────────────┐
          │    transactions    │────▶│   time_dimension │
          │  transaction_id PK │     │  date         PK │
          │  customer_id    FK │     │  day/month/year  │
          │  product_id     FK │     │  quarter         │
          │  order_date     FK │     │  is_weekend      │
          │  prices/payment    │     │  is_festival     │
          │  delivery/return   │     └──────────────────┘
          └────────────────────┘
```

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TABLE 1 — customers
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
customer_cols = [
    'customer_id', 'customer_city', 'customer_state',
    'customer_tier', 'customer_age_group', 'customer_spending_tier',
    'is_prime_member'
]
customer_cols = [c for c in customer_cols if c in master_df.columns]

customers_table = (
    master_df[customer_cols]
    .drop_duplicates(subset=['customer_id'])
    .reset_index(drop=True)
)
if 'is_prime_member' in customers_table.columns:
    customers_table['is_prime_member'] = customers_table['is_prime_member'].astype(int)

print(f'✅ customers      : {len(customers_table):,} unique rows')
print(f'   Columns : {list(customers_table.columns)}')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TABLE 2 — products
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
product_cols = [
    'product_id', 'product_name', 'category', 'subcategory',
    'brand', 'product_rating', 'is_prime_eligible'
]
product_cols = [c for c in product_cols if c in master_df.columns]

products_table = (
    master_df[product_cols]
    .drop_duplicates(subset=['product_id'])
    .reset_index(drop=True)
)

# Merge extra columns from product catalog
catalog_extra = ['base_price_2015', 'weight_kg', 'launch_year', 'model']
catalog_extra = [c for c in catalog_extra if c in products_df.columns]
if catalog_extra and 'product_id' in products_df.columns:
    products_table = products_table.merge(
        products_df[['product_id'] + catalog_extra],
        on='product_id', how='left'
    )
if 'is_prime_eligible' in products_table.columns:
    products_table['is_prime_eligible'] = products_table['is_prime_eligible'].astype(int)

print(f'\n✅ products       : {len(products_table):,} unique rows')
print(f'   Columns : {list(products_table.columns)}')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TABLE 3 — time_dimension
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
dates = master_df['order_date']

time_base = pd.DataFrame({
    'date'        : dates.dt.strftime('%Y-%m-%d'),
    'day'         : dates.dt.day,
    'month'       : dates.dt.month,
    'month_name'  : dates.dt.strftime('%B'),
    'quarter'     : dates.dt.quarter,
    'year'        : dates.dt.year,
    'day_of_week' : dates.dt.dayofweek,
    'day_name'    : dates.dt.strftime('%A'),
    'is_weekend'  : dates.dt.dayofweek.isin([5, 6]).astype(int),
    'week_of_year': dates.dt.isocalendar().week.astype(int),
})

# Attach festival info
for col in ['is_festival_sale', 'festival_name']:
    if col in master_df.columns:
        time_base[col] = master_df[col].values
if 'is_festival_sale' in time_base.columns:
    time_base['is_festival_sale'] = time_base['is_festival_sale'].astype(int)

time_dimension = (
    time_base
    .drop_duplicates(subset=['date'])
    .dropna(subset=['date'])
    .sort_values('date')
    .reset_index(drop=True)
)
print(f'\n✅ time_dimension : {len(time_dimension):,} unique dates')
print(f'   Range   : {time_dimension["date"].min()} → {time_dimension["date"].max()}')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TABLE 4 — transactions (main fact table — keep all remaining columns)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
drop_from_txn = [
    'customer_city', 'customer_state', 'customer_tier',
    'customer_age_group', 'customer_spending_tier',
    'product_name', 'category', 'subcategory', 'brand',
    'product_rating', 'is_prime_eligible',
    'order_month', 'order_quarter',       # these come from time_dimension
]
txn_cols = [c for c in master_df.columns if c not in drop_from_txn]
transactions_table = master_df[txn_cols].copy()
transactions_table['order_date'] = transactions_table['order_date'].dt.strftime('%Y-%m-%d')

# Bool → int for SQLite
for col in ['is_prime_member', 'is_festival_sale', 'is_weekend']:
    if col in transactions_table.columns:
        transactions_table[col] = transactions_table[col].astype(int)

print(f'\n✅ transactions   : {len(transactions_table):,} rows | {transactions_table.shape[1]} columns')
print()
print('━'*55)
print('ALL 4 TABLES READY')
print('━'*55)
print(f'  customers      : {len(customers_table):>8,} rows')
print(f'  products       : {len(products_table):>8,} rows')
print(f'  time_dimension : {len(time_dimension):>8,} rows')
print(f'  transactions   : {len(transactions_table):>8,} rows')

✅ customers      : 354,969 unique rows
   Columns : ['customer_id', 'customer_city', 'customer_state', 'customer_tier', 'customer_age_group', 'customer_spending_tier', 'is_prime_member']

✅ products       : 2,004 unique rows
   Columns : ['product_id', 'product_name', 'category', 'subcategory', 'brand', 'product_rating', 'is_prime_eligible', 'base_price_2015', 'weight_kg', 'launch_year', 'model']

✅ time_dimension : 4,015 unique dates
   Range   : 2015-01-01 → 2025-12-31

✅ transactions   : 1,122,000 rows | 21 columns

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ALL 4 TABLES READY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  customers      :  354,969 rows
  products       :    2,004 rows
  time_dimension :    4,015 rows
  transactions   : 1,122,000 rows


## ⑤ Create Schema & Load Tables into SQLite

In [5]:
# ── Drop existing tables for a fresh load ─────────────────────────────────────
with engine.connect() as conn:
    for tbl in ['transactions', 'customers', 'products', 'time_dimension']:
        conn.execute(text(f'DROP TABLE IF EXISTS {tbl}'))
    conn.commit()
print('✅ Existing tables dropped')

# ── Loader helper ─────────────────────────────────────────────────────────────
def load_table(df, table_name, chunksize=10_000):
    t0 = time.time()
    df.to_sql(
        table_name, con=engine,
        if_exists='replace', index=False,
        chunksize=10_000, method=None         # ← method=None uses safe single-row inserts
    )
    elapsed = time.time() - t0
    print(f'  ✅ {table_name:<22} {len(df):>8,} rows  loaded in {elapsed:.1f}s')

print()
print('Loading tables into SQLite...')
print('━'*55)
load_table(customers_table,    'customers')
load_table(products_table,     'products')
load_table(time_dimension,     'time_dimension')
load_table(transactions_table, 'transactions')
print('━'*55)

db_size_mb = os.path.getsize(DB_PATH) / 1e6
print(f'✅ All tables loaded!  DB size: {db_size_mb:.1f} MB')

✅ Existing tables dropped

Loading tables into SQLite...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ customers               354,969 rows  loaded in 3.3s
  ✅ products                  2,004 rows  loaded in 0.0s
  ✅ time_dimension            4,015 rows  loaded in 0.1s
  ✅ transactions           1,122,000 rows  loaded in 27.7s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ All tables loaded!  DB size: 407.5 MB


## ⑥ Create Indexes for Performance

In [6]:
indexes = [
    # transactions — most heavily queried
    ('idx_txn_order_date',     'CREATE INDEX IF NOT EXISTS idx_txn_order_date     ON transactions(order_date)'),
    ('idx_txn_order_year',     'CREATE INDEX IF NOT EXISTS idx_txn_order_year     ON transactions(order_year)'),
    ('idx_txn_customer_id',    'CREATE INDEX IF NOT EXISTS idx_txn_customer_id    ON transactions(customer_id)'),
    ('idx_txn_product_id',     'CREATE INDEX IF NOT EXISTS idx_txn_product_id     ON transactions(product_id)'),
    ('idx_txn_payment',        'CREATE INDEX IF NOT EXISTS idx_txn_payment        ON transactions(payment_method)'),
    ('idx_txn_return',         'CREATE INDEX IF NOT EXISTS idx_txn_return         ON transactions(return_status)'),
    ('idx_txn_is_festival',    'CREATE INDEX IF NOT EXISTS idx_txn_is_festival    ON transactions(is_festival_sale)'),
    ('idx_txn_is_prime',       'CREATE INDEX IF NOT EXISTS idx_txn_is_prime       ON transactions(is_prime_member)'),
    ('idx_txn_year_month',     'CREATE INDEX IF NOT EXISTS idx_txn_year_month     ON transactions(order_year, order_date)'),
    # customers
    ('idx_cust_city',          'CREATE INDEX IF NOT EXISTS idx_cust_city          ON customers(customer_city)'),
    ('idx_cust_state',         'CREATE INDEX IF NOT EXISTS idx_cust_state         ON customers(customer_state)'),
    ('idx_cust_tier',          'CREATE INDEX IF NOT EXISTS idx_cust_tier          ON customers(customer_tier)'),
    # products
    ('idx_prod_category',      'CREATE INDEX IF NOT EXISTS idx_prod_category      ON products(category)'),
    ('idx_prod_subcategory',   'CREATE INDEX IF NOT EXISTS idx_prod_subcategory   ON products(subcategory)'),
    ('idx_prod_brand',         'CREATE INDEX IF NOT EXISTS idx_prod_brand         ON products(brand)'),
    # time_dimension
    ('idx_time_year',          'CREATE INDEX IF NOT EXISTS idx_time_year          ON time_dimension(year)'),
    ('idx_time_month',         'CREATE INDEX IF NOT EXISTS idx_time_month         ON time_dimension(month)'),
    ('idx_time_quarter',       'CREATE INDEX IF NOT EXISTS idx_time_quarter       ON time_dimension(quarter)'),
]

print('Creating indexes...')
print('━'*55)
with engine.connect() as conn:
    for name, ddl in indexes:
        try:
            conn.execute(text(ddl))
            print(f'  ✅ {name}')
        except Exception as e:
            print(f'  ⚠️  {name} — skipped: {e}')
    conn.commit()
print('━'*55)
print(f'✅ {len(indexes)} indexes created!')

Creating indexes...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ idx_txn_order_date
  ✅ idx_txn_order_year
  ✅ idx_txn_customer_id
  ✅ idx_txn_product_id
  ✅ idx_txn_payment
  ✅ idx_txn_return
  ✅ idx_txn_is_festival
  ✅ idx_txn_is_prime
  ✅ idx_txn_year_month
  ✅ idx_cust_city
  ✅ idx_cust_state
  ✅ idx_cust_tier
  ✅ idx_prod_category
  ✅ idx_prod_subcategory
  ✅ idx_prod_brand
  ✅ idx_time_year
  ✅ idx_time_month
  ✅ idx_time_quarter
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ 18 indexes created!


## ⑦ Validation & Row Count Checks

In [7]:
# ── Row counts ────────────────────────────────────────────────────────────────
print('━'*55)
print('VALIDATION — Row Counts')
print('━'*55)
with engine.connect() as conn:
    for tbl in ['customers', 'products', 'time_dimension', 'transactions']:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {tbl}')).fetchone()[0]
        print(f'  {tbl:<22} : {count:>8,} rows')

# ── Null checks on key columns ────────────────────────────────────────────────
print()
print('━'*55)
print('VALIDATION — Null Checks (transactions)')
print('━'*55)
key_cols = ['transaction_id', 'customer_id', 'product_id',
            'order_date', 'final_amount_inr', 'payment_method']
with engine.connect() as conn:
    for col in key_cols:
        try:
            nulls = conn.execute(
                text(f'SELECT COUNT(*) FROM transactions WHERE {col} IS NULL')
            ).fetchone()[0]
            status = '✅' if nulls == 0 else '⚠️ '
            print(f'  {status} {col:<28} nulls: {nulls:,}')
        except Exception as e:
            print(f'  ⚠️  {col} not found: {e}')

# ── Year distribution ─────────────────────────────────────────────────────────
print()
print('━'*55)
print('VALIDATION — Orders per Year')
print('━'*55)
with engine.connect() as conn:
    rows = conn.execute(text(
        'SELECT order_year, COUNT(*) AS orders FROM transactions GROUP BY order_year ORDER BY order_year'
    )).fetchall()
    max_orders = max(r[1] for r in rows)
    for r in rows:
        bar = '█' * int(r[1] / max_orders * 35)
        print(f'  {r[0]} : {r[1]:>8,}  {bar}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
VALIDATION — Row Counts
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  customers              :  354,969 rows
  products               :    2,004 rows
  time_dimension         :    4,015 rows
  transactions           : 1,122,000 rows

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
VALIDATION — Null Checks (transactions)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ transaction_id               nulls: 0
  ✅ customer_id                  nulls: 0
  ✅ product_id                   nulls: 0
  ✅ order_date                   nulls: 0
  ✅ final_amount_inr             nulls: 0
  ✅ payment_method               nulls: 0

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
VALIDATION — Orders per Year
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  2015 :   33,000  ████████
  2016 :   55,000  █████████████
  2017 :   77,000  ██████████████████
  2018 :   99,000  ████████████████████████
  2019 : 

## ⑧ Analytics Queries — 15 Dashboard KPI Queries

> These are the exact queries the Streamlit app will use — verified here first

In [8]:
# ── Query runner helper ───────────────────────────────────────────────────────
def run_query(sql, label=''):
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    if label:
        print(f'\n━━━ {label} ━━━')
        print(df.to_string(index=False))
    return df

In [9]:
# ── KPI 1: Executive Summary ──────────────────────────────────────────────────
run_query("""
    SELECT
        COUNT(*)                               AS total_transactions,
        COUNT(DISTINCT customer_id)            AS active_customers,
        ROUND(SUM(final_amount_inr)/1e9, 3)   AS revenue_bn,
        ROUND(AVG(final_amount_inr), 2)        AS avg_order_value
    FROM transactions
""", 'KPI 1 — Executive Summary')


━━━ KPI 1 — Executive Summary ━━━
 total_transactions  active_customers  revenue_bn  avg_order_value
            1122000            354969      76.498         68180.34


,total_transactions,active_customers,revenue_bn,avg_order_value
0,1122000,354969,76.498,68180.34


In [10]:
# ── KPI 2: Yearly Revenue Trend (EDA Q1, Dashboard Q6) ───────────────────────
run_query("""
    SELECT
        order_year,
        COUNT(*)                               AS total_orders,
        ROUND(SUM(final_amount_inr)/1e7, 2)   AS revenue_cr,
        ROUND(AVG(final_amount_inr), 2)        AS avg_order_value
    FROM transactions
    GROUP BY order_year
    ORDER BY order_year
""", 'KPI 2 — Yearly Revenue Trend')


━━━ KPI 2 — Yearly Revenue Trend ━━━
 order_year  total_orders  revenue_cr  avg_order_value
       2015         33000      213.05         64560.58
       2016         55000      357.99         65089.88
       2017         77000      548.00         71168.23
       2018         99000      721.33         72861.37
       2019        121000      856.35         70772.35
       2020        143000     1181.36         82612.86
       2021        137500     1093.41         79520.81
       2022        132000      848.97         64316.14
       2023        126500      767.38         60662.63
       2024        121000      678.82         56100.90
       2025         77000      383.17         49762.30


,order_year,total_orders,revenue_cr,avg_order_value
0,2015,33000,213.05,64560.58
1,2016,55000,357.99,65089.88
2,2017,77000,548.00,71168.23
3,2018,99000,721.33,72861.37
4,2019,121000,856.35,70772.35
5,2020,143000,1181.36,82612.86
6,2021,137500,1093.41,79520.81
7,2022,132000,848.97,64316.14
8,2023,126500,767.38,60662.63
9,2024,121000,678.82,56100.90


In [11]:
# ── KPI 3: Monthly Seasonality Heatmap (EDA Q2) ───────────────────────────────
run_query("""
    SELECT
        order_year,
        CAST(strftime('%m', order_date) AS INTEGER)  AS order_month,
        ROUND(SUM(final_amount_inr)/1e7, 2)           AS revenue_cr,
        COUNT(*)                                       AS orders
    FROM transactions
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
    LIMIT 15
""", 'KPI 3 — Monthly Seasonality (sample)')


━━━ KPI 3 — Monthly Seasonality (sample) ━━━
 order_year  order_month  revenue_cr  orders
       2015            1       16.20    2410
       2015            2       14.30    2100
       2015            3       14.58    2407
       2015            4       17.75    2411
       2015            5       15.71    2701
       2015            6       13.90    2401
       2015            7       15.53    2712
       2015            8       17.02    2405
       2015            9       17.06    2706
       2015           10       20.99    3881
       2015           11       23.06    3281
       2015           12       26.96    3585
       2016            1       26.92    4009
       2016            2       24.48    3512
       2016            3       24.49    4016


,order_year,order_month,revenue_cr,orders
0,2015,1,16.20,2410
1,2015,2,14.30,2100
2,2015,3,14.58,2407
3,2015,4,17.75,2411
4,2015,5,15.71,2701
5,2015,6,13.90,2401
6,2015,7,15.53,2712
7,2015,8,17.02,2405
8,2015,9,17.06,2706
9,2015,10,20.99,3881


In [12]:
# ── KPI 4: Category Performance (EDA Q5, Dashboard Q7) ───────────────────────
run_query("""
    SELECT
        p.subcategory,
        p.category,
        COUNT(t.transaction_id)                AS total_orders,
        ROUND(SUM(t.final_amount_inr)/1e7, 2) AS revenue_cr,
        ROUND(AVG(t.final_amount_inr), 2)      AS avg_order_value,
        ROUND(AVG(t.discount_percent), 2)      AS avg_discount_pct
    FROM transactions t
    JOIN products p ON t.product_id = p.product_id
    GROUP BY p.subcategory, p.category
    ORDER BY revenue_cr DESC
""", 'KPI 4 — Category Performance')


━━━ KPI 4 — Category Performance ━━━
       subcategory    category  total_orders  revenue_cr  avg_order_value  avg_discount_pct
       Smartphones Electronics        823074     5587.06         67880.47             17.42
           Laptops Electronics         87901      935.23        106395.87             17.37
           Tablets Electronics         69927      505.92         72350.02             17.43
       Smart Watch Electronics         74335      319.79         43020.27             17.44
Tv & Entertainment Electronics         16390      191.27        116700.02             17.31
             Audio Electronics         50373      110.55         21947.07             17.52


,subcategory,category,total_orders,revenue_cr,avg_order_value,avg_discount_pct
0,Smartphones,Electronics,823074,5587.06,67880.47,17.42
1,Laptops,Electronics,87901,935.23,106395.87,17.37
2,Tablets,Electronics,69927,505.92,72350.02,17.43
3,Smart Watch,Electronics,74335,319.79,43020.27,17.44
4,Tv & Entertainment,Electronics,16390,191.27,116700.02,17.31
5,Audio,Electronics,50373,110.55,21947.07,17.52


In [13]:
# ── KPI 5: Payment Method Evolution (EDA Q4, Dashboard Q22) ──────────────────
run_query("""
    SELECT
        order_year,
        payment_method,
        COUNT(*)  AS orders,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY order_year), 2) AS market_share_pct
    FROM transactions
    GROUP BY order_year, payment_method
    ORDER BY order_year, orders DESC
""", 'KPI 5 — Payment Method Evolution')


━━━ KPI 5 — Payment Method Evolution ━━━
 order_year   payment_method  orders  market_share_pct
       2015 Cash on Delivery   24835             75.26
       2015      Credit Card    3989             12.09
       2015       Debit Card    2588              7.84
       2015      Net Banking    1588              4.81
       2016 Cash on Delivery   38534             70.06
       2016      Credit Card    7754             14.10
       2016       Debit Card    5433              9.88
       2016      Net Banking    2145              3.90
       2016              UPI    1134              2.06
       2017 Cash on Delivery   46160             59.95
       2017      Credit Card   12357             16.05
       2017       Debit Card    9211             11.96
       2017              UPI    5447              7.07
       2017      Net Banking    3825              4.97
       2018 Cash on Delivery   49466             49.97
       2018      Credit Card   16798             16.97
       2018       Debit

,order_year,payment_method,orders,market_share_pct
0,2015,Cash on Delivery,24835,75.26
1,2015,Credit Card,3989,12.09
2,2015,Debit Card,2588,7.84
3,2015,Net Banking,1588,4.81
4,2016,Cash on Delivery,38534,70.06
...,...,...,...,...
59,2025,Cash on Delivery,6207,8.06
60,2025,Debit Card,6197,8.05
61,2025,BNPL,5238,6.80
62,2025,Net Banking,2270,2.95


In [14]:
# ── KPI 6: Prime vs Non-Prime (EDA Q6, Dashboard Q13) ────────────────────────
run_query("""
    SELECT
        is_prime_member,
        COUNT(*)                               AS total_orders,
        COUNT(DISTINCT customer_id)            AS unique_customers,
        ROUND(SUM(final_amount_inr)/1e7, 2)   AS revenue_cr,
        ROUND(AVG(final_amount_inr), 2)        AS avg_order_value,
        ROUND(AVG(customer_rating), 2)         AS avg_rating,
        ROUND(AVG(delivery_days), 2)           AS avg_delivery_days
    FROM transactions
    GROUP BY is_prime_member
""", 'KPI 6 — Prime vs Non-Prime')


━━━ KPI 6 — Prime vs Non-Prime ━━━
 is_prime_member  total_orders  unique_customers  revenue_cr  avg_order_value  avg_rating  avg_delivery_days
               0        694996            194307     4310.63         62023.84        4.35               4.32
               1        427004            160662     3339.20         78200.72        4.40               1.72


,is_prime_member,total_orders,unique_customers,revenue_cr,avg_order_value,avg_rating,avg_delivery_days
0,0,694996,194307,4310.63,62023.84,4.35,4.32
1,1,427004,160662,3339.20,78200.72,4.40,1.72


In [15]:
# ── KPI 7: Geographic — Top Cities (EDA Q7, Dashboard Q8) ────────────────────
run_query("""
    SELECT
        c.customer_state,
        c.customer_city,
        c.customer_tier,
        COUNT(t.transaction_id)                AS total_orders,
        ROUND(SUM(t.final_amount_inr)/1e7, 2) AS revenue_cr
    FROM transactions t
    JOIN customers c ON t.customer_id = c.customer_id
    GROUP BY c.customer_state, c.customer_city, c.customer_tier
    ORDER BY revenue_cr DESC
    LIMIT 20
""", 'KPI 7 — Top 20 Cities by Revenue')


━━━ KPI 7 — Top 20 Cities by Revenue ━━━
customer_state customer_city customer_tier  total_orders  revenue_cr
   Maharashtra        Mumbai         Metro        140900     1059.93
         Delhi     New Delhi         Metro        122860      923.47
     Karnataka     Bengaluru         Metro        102006      763.89
    Tamil Nadu       Chennai         Metro         84335      638.61
   West Bengal       Kolkata         Metro         66547      502.91
   Maharashtra          Pune         Tier1         67392      442.31
     Telangana     Hyderabad         Metro         44346      331.99
       Gujarat     Ahmedabad         Tier1         50596      328.33
       Gujarat         Surat         Tier1         40177      265.22
     Rajasthan        Jaipur         Tier1         40251      263.16
   Maharashtra        Nagpur         Tier1         37026      243.65
 Uttar Pradesh       Lucknow         Tier1         33903      223.25
 Uttar Pradesh        Kanpur         Tier1         34157     

,customer_state,customer_city,customer_tier,total_orders,revenue_cr
0,Maharashtra,Mumbai,Metro,140900,1059.93
1,Delhi,New Delhi,Metro,122860,923.47
2,Karnataka,Bengaluru,Metro,102006,763.89
3,Tamil Nadu,Chennai,Metro,84335,638.61
4,West Bengal,Kolkata,Metro,66547,502.91
5,Maharashtra,Pune,Tier1,67392,442.31
6,Telangana,Hyderabad,Metro,44346,331.99
7,Gujarat,Ahmedabad,Tier1,50596,328.33
8,Gujarat,Surat,Tier1,40177,265.22
9,Rajasthan,Jaipur,Tier1,40251,263.16


In [16]:
# ── KPI 8: Festival Sales Impact (EDA Q8, Dashboard Q9) ──────────────────────
run_query("""
    SELECT
        is_festival_sale,
        festival_name,
        COUNT(*)                               AS total_orders,
        ROUND(SUM(final_amount_inr)/1e7, 2)   AS revenue_cr,
        ROUND(AVG(final_amount_inr), 2)        AS avg_order_value,
        ROUND(AVG(discount_percent), 2)        AS avg_discount_pct
    FROM transactions
    GROUP BY is_festival_sale, festival_name
    ORDER BY revenue_cr DESC
""", 'KPI 8 — Festival Sales Impact')


━━━ KPI 8 — Festival Sales Impact ━━━
 is_festival_sale                festival_name  total_orders  revenue_cr  avg_order_value  avg_discount_pct
                0                         None        773820     5996.40         77490.95              6.13
                1               Back to School         84975      405.30         47696.96             42.53
                1                  Diwali Sale         83199      393.95         47350.54             42.53
                1 Amazon Great Indian Festival         52828      250.60         47437.59             42.53
                1                  Summer Sale         44712      210.87         47162.26             42.51
                1                Holi Festival         39409      187.59         47600.41             42.53
                1            Republic Day Sale         20085       95.54         47569.29             42.57
                1               Valentine Sale         14042       66.24         47172.80        

,is_festival_sale,festival_name,total_orders,revenue_cr,avg_order_value,avg_discount_pct
0,0,None,773820,5996.40,77490.95,6.13
1,1,Back to School,84975,405.30,47696.96,42.53
2,1,Diwali Sale,83199,393.95,47350.54,42.53
3,1,Amazon Great Indian Festival,52828,250.60,47437.59,42.53
4,1,Summer Sale,44712,210.87,47162.26,42.51
5,1,Holi Festival,39409,187.59,47600.41,42.53
6,1,Republic Day Sale,20085,95.54,47569.29,42.57
7,1,Valentine Sale,14042,66.24,47172.80,42.39
8,1,Prime Day,8930,43.33,48516.90,42.36


In [17]:
# ── KPI 9: Delivery & Operations (EDA Q11, Dashboard Q21) ────────────────────
run_query("""
    SELECT
        ROUND(AVG(delivery_days), 2)                                         AS avg_delivery_days,
        ROUND(100.0 * SUM(CASE WHEN delivery_days <= 3 THEN 1 ELSE 0 END)
              / COUNT(*), 2)                                                  AS on_time_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN return_status = 'Returned'  THEN 1 ELSE 0 END)
              / COUNT(*), 2)                                                  AS return_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN return_status = 'Cancelled' THEN 1 ELSE 0 END)
              / COUNT(*), 2)                                                  AS cancellation_rate_pct
    FROM transactions
""", 'KPI 9 — Delivery & Operations')


━━━ KPI 9 — Delivery & Operations ━━━
 avg_delivery_days  on_time_rate_pct  return_rate_pct  cancellation_rate_pct
              3.33             57.05             7.02                   2.31


,avg_delivery_days,on_time_rate_pct,return_rate_pct,cancellation_rate_pct
0,3.33,57.05,7.02,2.31


In [18]:
# ── KPI 10: Brand Performance (EDA Q13, Dashboard Q17) ───────────────────────
run_query("""
    SELECT
        p.brand,
        COUNT(t.transaction_id)                AS total_orders,
        ROUND(SUM(t.final_amount_inr)/1e7, 2) AS revenue_cr,
        ROUND(AVG(t.final_amount_inr), 2)      AS avg_order_value,
        ROUND(AVG(p.product_rating), 2)        AS avg_rating,
        ROUND(100.0 * SUM(CASE WHEN t.return_status='Returned' THEN 1 ELSE 0 END)
              / COUNT(*), 2)                   AS return_rate_pct
    FROM transactions t
    JOIN products p ON t.product_id = p.product_id
    GROUP BY p.brand
    ORDER BY revenue_cr DESC
    LIMIT 15
""", 'KPI 10 — Top 15 Brands')


━━━ KPI 10 — Top 15 Brands ━━━
    brand  total_orders  revenue_cr  avg_order_value  avg_rating  return_rate_pct
  Samsung        221606     2052.10         92601.06        3.97             6.53
    Apple        111256     1623.96        145965.97        3.99             6.03
  OnePlus        173445     1225.92         70680.68        4.02             6.61
   Xiaomi        155364      542.24         34901.15        3.97             7.53
   Realme         84272      300.08         35608.74        3.96             7.48
     Vivo         70802      233.61         32994.34        3.98             7.74
     Oppo         70890      225.84         31857.50        3.97             7.63
   Lenovo         22625      207.49         91707.47        3.98             6.99
Alienware         14612      165.17        113037.08        4.15             6.27
     ASUS         12993      145.15        111717.21        4.00             7.37
     Acer         13894      141.03        101505.31        4.05  

,brand,total_orders,revenue_cr,avg_order_value,avg_rating,return_rate_pct
0,Samsung,221606,2052.10,92601.06,3.97,6.53
1,Apple,111256,1623.96,145965.97,3.99,6.03
2,OnePlus,173445,1225.92,70680.68,4.02,6.61
3,Xiaomi,155364,542.24,34901.15,3.97,7.53
4,Realme,84272,300.08,35608.74,3.96,7.48
5,Vivo,70802,233.61,32994.34,3.98,7.74
6,Oppo,70890,225.84,31857.50,3.97,7.63
7,Lenovo,22625,207.49,91707.47,3.98,6.99
8,Alienware,14612,165.17,113037.08,4.15,6.27
9,ASUS,12993,145.15,111717.21,4.00,7.37


In [19]:
# ── KPI 11: RFM Base for Segmentation (EDA Q3, Dashboard Q11) ────────────────
run_query("""
    SELECT
        customer_id,
        MAX(order_date)                        AS last_order_date,
        COUNT(transaction_id)                  AS frequency,
        ROUND(SUM(final_amount_inr), 2)        AS monetary
    FROM transactions
    GROUP BY customer_id
    ORDER BY monetary DESC
    LIMIT 10
""", 'KPI 11 — RFM Sample (Top 10 by Spend)')


━━━ KPI 11 — RFM Sample (Top 10 by Spend) ━━━
       customer_id last_order_date  frequency   monetary
CUST_2015_00003928      2021-12-28         27 3757087.02
CUST_2016_00007974      2023-04-10         17 3708826.25
CUST_2016_00001495      2023-12-01         20 3533620.11
CUST_2015_00009716      2021-02-25         19 3134243.67
CUST_2018_00007329      2024-07-28         17 3132390.03
CUST_2017_00014541      2025-12-24         23 3102679.55
CUST_2015_00009173      2021-07-11         18 3073245.76
CUST_2015_00002706      2021-08-17         16 3026726.21
CUST_2020_00050279      2023-09-12          8 3005986.14
CUST_2015_00004117      2024-05-27         15 3005482.80


,customer_id,last_order_date,frequency,monetary
0,CUST_2015_00003928,2021-12-28,27,3757087.02
1,CUST_2016_00007974,2023-04-10,17,3708826.25
2,CUST_2016_00001495,2023-12-01,20,3533620.11
3,CUST_2015_00009716,2021-02-25,19,3134243.67
4,CUST_2018_00007329,2024-07-28,17,3132390.03
5,CUST_2017_00014541,2025-12-24,23,3102679.55
6,CUST_2015_00009173,2021-07-11,18,3073245.76
7,CUST_2015_00002706,2021-08-17,16,3026726.21
8,CUST_2020_00050279,2023-09-12,8,3005986.14
9,CUST_2015_00004117,2024-05-27,15,3005482.80


In [20]:
# ── KPI 12: Discount Bands vs Revenue (EDA Q15, Dashboard Q10) ───────────────
run_query("""
    SELECT
        CASE
            WHEN discount_percent = 0       THEN '0% No Discount'
            WHEN discount_percent <= 10     THEN '1-10%'
            WHEN discount_percent <= 20     THEN '11-20%'
            WHEN discount_percent <= 30     THEN '21-30%'
            WHEN discount_percent <= 50     THEN '31-50%'
            ELSE '50%+'
        END AS discount_band,
        COUNT(*)                               AS orders,
        ROUND(SUM(final_amount_inr)/1e7, 2)   AS revenue_cr,
        ROUND(AVG(final_amount_inr), 2)        AS avg_order_value
    FROM transactions
    GROUP BY discount_band
    ORDER BY MIN(discount_percent)
""", 'KPI 12 — Discount Bands vs Revenue')


━━━ KPI 12 — Discount Bands vs Revenue ━━━
 discount_band  orders  revenue_cr  avg_order_value
0% No Discount  503281     4156.64         82590.87
         1-10%   53771      411.32         76494.37
        11-20%  139743      972.40         69585.14
        21-30%  172079     1066.08         61952.75
        31-50%  126337      624.81         49456.11
          50%+  126789      418.58         33013.91


,discount_band,orders,revenue_cr,avg_order_value
0,0% No Discount,503281,4156.64,82590.87
1,1-10%,53771,411.32,76494.37
2,11-20%,139743,972.40,69585.14
3,21-30%,172079,1066.08,61952.75
4,31-50%,126337,624.81,49456.11
5,50%+,126789,418.58,33013.91


In [21]:
# ── KPI 13: Age Group Behaviour (EDA Q9, Dashboard Q15) ──────────────────────
run_query("""
    SELECT
        c.customer_age_group,
        COUNT(t.transaction_id)                AS total_orders,
        COUNT(DISTINCT t.customer_id)          AS unique_customers,
        ROUND(SUM(t.final_amount_inr)/1e7, 2) AS revenue_cr,
        ROUND(AVG(t.final_amount_inr), 2)      AS avg_spend
    FROM transactions t
    JOIN customers c ON t.customer_id = c.customer_id
    GROUP BY c.customer_age_group
    ORDER BY revenue_cr DESC
""", 'KPI 13 — Age Group Behaviour')


━━━ KPI 13 — Age Group Behaviour ━━━
customer_age_group  total_orders  unique_customers  revenue_cr  avg_spend
             26-35        345956            109763     2363.06   68305.19
             18-25        315145             99443     2147.71   68150.05
             36-45        197735             62480     1346.20   68080.99
              None        134904             42700      922.92   68413.25
             46-55         98435             31086      668.45   67907.94
               55+         29825              9497      201.49   67556.26


,customer_age_group,total_orders,unique_customers,revenue_cr,avg_spend
0,26-35,345956,109763,2363.06,68305.19
1,18-25,315145,99443,2147.71,68150.05
2,36-45,197735,62480,1346.20,68080.99
3,None,134904,42700,922.92,68413.25
4,46-55,98435,31086,668.45,67907.94
5,55+,29825,9497,201.49,67556.26


In [22]:
# ── KPI 14: Return Rate by Subcategory (EDA Q12, Dashboard Q23) ──────────────
run_query("""
    SELECT
        p.subcategory,
        COUNT(t.transaction_id)  AS total_orders,
        SUM(CASE WHEN t.return_status = 'Returned'  THEN 1 ELSE 0 END) AS returns,
        SUM(CASE WHEN t.return_status = 'Cancelled' THEN 1 ELSE 0 END) AS cancellations,
        ROUND(100.0 * SUM(CASE WHEN t.return_status = 'Returned' THEN 1 ELSE 0 END)
              / COUNT(*), 2)     AS return_rate_pct,
        ROUND(AVG(t.customer_rating), 2)         AS avg_rating
    FROM transactions t
    JOIN products p ON t.product_id = p.product_id
    GROUP BY p.subcategory
    ORDER BY return_rate_pct DESC
""", 'KPI 14 — Return Rate by Subcategory')


━━━ KPI 14 — Return Rate by Subcategory ━━━
       subcategory  total_orders  returns  cancellations  return_rate_pct  avg_rating
             Audio         50373     4037           1343             8.01        4.31
       Smart Watch         74335     5477           1824             7.37        4.35
           Tablets         69927     4896           1619             7.00        4.38
       Smartphones        823074    57288          18833             6.96        4.37
           Laptops         87901     6059           2001             6.89        4.41
Tv & Entertainment         16390      956            305             5.83        4.42


,subcategory,total_orders,returns,cancellations,return_rate_pct,avg_rating
0,Audio,50373,4037,1343,8.01,4.31
1,Smart Watch,74335,5477,1824,7.37,4.35
2,Tablets,69927,4896,1619,7.00,4.38
3,Smartphones,823074,57288,18833,6.96,4.37
4,Laptops,87901,6059,2001,6.89,4.41
5,Tv & Entertainment,16390,956,305,5.83,4.42


In [23]:
# ── KPI 15: Cohort Retention (EDA Q14, Dashboard Q14) ────────────────────────
run_query("""
    SELECT
        strftime('%Y', MIN(order_date))        AS cohort_year,
        order_year                              AS active_year,
        COUNT(DISTINCT customer_id)            AS active_customers,
        ROUND(SUM(final_amount_inr)/1e7, 2)   AS revenue_cr
    FROM transactions
    GROUP BY strftime('%Y', order_date), order_year
    ORDER BY cohort_year, active_year
    LIMIT 20
""", 'KPI 15 — Cohort Retention Sample')


━━━ KPI 15 — Cohort Retention Sample ━━━
cohort_year  active_year  active_customers  revenue_cr
       2015         2015             11222      213.05
       2016         2016             26273      357.99
       2017         2017             38789      548.00
       2018         2018             49917      721.33
       2019         2019             61068      856.35
       2020         2020             71997     1181.36
       2021         2021             69375     1093.41
       2022         2022             66605      848.97
       2023         2023             63790      767.38
       2024         2024             60947      678.82
       2025         2025             38826      383.17


,cohort_year,active_year,active_customers,revenue_cr
0,2015,2015,11222,213.05
1,2016,2016,26273,357.99
2,2017,2017,38789,548.00
3,2018,2018,49917,721.33
4,2019,2019,61068,856.35
5,2020,2020,71997,1181.36
6,2021,2021,69375,1093.41
7,2022,2022,66605,848.97
8,2023,2023,63790,767.38
9,2024,2024,60947,678.82


## ⑨ Connection Test — Streamlit Pattern

> This is exactly what `utils/db.py` will look like in the Streamlit app

In [24]:
DB_ABS_PATH = os.path.abspath(DB_PATH)

print('━'*60)
print('utils/db.py — STREAMLIT CONNECTION CODE')
print('━'*60)
print(f'''
# utils/db.py
import pandas as pd
import streamlit as st
from sqlalchemy import create_engine, text

DB_PATH = r"{DB_ABS_PATH}"

@st.cache_resource
def get_engine():
    """Create and cache a SQLAlchemy engine."""
    return create_engine(f"sqlite:///{{DB_PATH}}", echo=False)

@st.cache_data(ttl=300)
def run_query(sql: str, params: dict = None) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame. Results cached for 5 minutes."""
    engine = get_engine()
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)
''')

# ── Live connection test ──────────────────────────────────────────────────────
print('━'*60)
print('LIVE TEST — simulating Streamlit query call')
print('━'*60)
test_engine = create_engine(f'sqlite:///{DB_ABS_PATH}', echo=False)
with test_engine.connect() as conn:
    test_df = pd.read_sql(text(
        'SELECT order_year, ROUND(SUM(final_amount_inr)/1e7,2) AS revenue_cr, COUNT(*) AS orders '
        'FROM transactions GROUP BY order_year ORDER BY order_year'
    ), conn)
print(test_df.to_string(index=False))
print()
print('✅ Streamlit connection pattern verified!')
print(f'   DB Path : {DB_ABS_PATH}')
print(f'   DB Size : {os.path.getsize(DB_ABS_PATH)/1e6:.1f} MB')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
utils/db.py — STREAMLIT CONNECTION CODE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# utils/db.py
import pandas as pd
import streamlit as st
from sqlalchemy import create_engine, text

DB_PATH = r"/home/ganesaperumal/Documents/DS/amazon_india_analytics/data/amazon_india.db"

@st.cache_resource
def get_engine():
    """Create and cache a SQLAlchemy engine."""
    return create_engine(f"sqlite:///{DB_PATH}", echo=False)

@st.cache_data(ttl=300)
def run_query(sql: str, params: dict = None) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame. Results cached for 5 minutes."""
    engine = get_engine()
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LIVE TEST — simulating Streamlit query call
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 order_year  revenue_cr  orders
       2015  

## ✅ Step 3 Complete — SQL Integration Summary

| Table | Rows | Key Columns |
|-------|------|-------------|
| `transactions` | ~1M | transaction_id, customer_id, product_id, order_date, final_amount_inr, payment_method, delivery_days, return_status |
| `customers` | ~355K | customer_id, city, state, tier, age_group, is_prime_member |
| `products` | ~2K | product_id, name, category, subcategory, brand, rating |
| `time_dimension` | ~3,650 | date, day, month, quarter, year, is_weekend, is_festival_sale |

**18 indexes** created on most-queried columns.

**15 KPI queries** verified and ready for Streamlit.

---